<a href="https://colab.research.google.com/github/Reginajose/Observatorio-Clima-Grande-desafio/blob/main/notebooks/historico/Transformacao_Filtros_nota_70_30.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Transformação dos Dados e Análises Estatísticas**



IMPORTANTO AS BIBILHOTECAS

In [1]:
import os
import pandas as pd
import numpy as np

# Leitura do Dataset Mestre (versão de 09/10: CAPAG set/2026 + colunas das decisões 1 e 2)
# 1º procura o arquivo local (pasta data/ do repositório ou upload no Colab);
# se não achar, lê direto do GitHub — assim roda no Colab sem precisar subir nada.
URL_GITHUB = 'https://raw.githubusercontent.com/Reginajose/Observatorio-Clima-Grande-desafio/main/data/dataset_mestre_observatorio.csv'
candidatos = ['data/dataset_mestre_observatorio.csv', '../data/dataset_mestre_observatorio.csv',
              'dataset_mestre_observatorio.csv', '/content/dataset_mestre_observatorio.csv']
caminho = next((c for c in candidatos if os.path.exists(c)), URL_GITHUB)

df_mestre = pd.read_csv(caminho, dtype={'Cod_IBGE_7': str}, encoding='utf-8-sig')

# Confere se é o mestre novo (09/10)
for col in ['criterio_fiscal_priorizacao', 'tem_dano_atlas', 'danos_humanos']:
    assert col in df_mestre.columns, f"Coluna {col} não encontrada: este é o dataset_mestre antigo. Baixe o novo do GitHub."

print(f"Dataset Mestre carregado: {df_mestre.shape[0]} municípios e {df_mestre.shape[1]} colunas.")

Dataset Mestre carregado: 5570 municípios e 20 colunas.


**1 - Aplicação dos Filtros do Funil**

In [2]:


# 1. Base Inicial
f0 = df_mestre.copy()

# 2. Filtro 1: Participou do SINISA
f1 = f0[
    (f0['participou_sinisa'] == True)
    | (
        f0['participou_sinisa']
        .astype(str)
        .str.strip()
        .str.upper()
        .isin(['TRUE', 'SIM', '1'])
    )
].copy()

# 3. Filtro 2: Domicílios em área de risco (> 0)
f1['qtd_domicilios_risco'] = pd.to_numeric(
    f1['qtd_domicilios_risco'], errors='coerce'
).fillna(0)
f1['pct_domicilios_risco'] = pd.to_numeric(
    f1['pct_domicilios_risco'], errors='coerce'
).fillna(0)
f2 = f1[f1['qtd_domicilios_risco'] > 0].copy()

# 4. Filtro 3: Fragilidade Fiscal / Sem Acesso a Crédito Bancário
# ATUALIZADO: Exclui apenas A, A+, B, B+ (Mantém C, D, n.d., n.e. e Nulos)
f2['capag_limpa'] = (
    f2['capag'].fillna('').astype(str).str.strip().str.upper()
)
f3 = f2[~f2['capag_limpa'].isin(['A', 'A+', 'B', 'B+'])].copy()

# 5. Filtro 4: Mapeamento de Risco Confirmado
f4 = f3[
    f3['mapeamento_area_risco']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.upper()
    .isin(['SIM', 'S', '1', 'TRUE'])
].copy()

# 6. Filtro 5 — Decisão 2D (07/10): lista federal de Críticos + dano em R$ registrado no Atlas 2021-2025
f5 = f4[
    (f4['municipio_critico'].fillna('').astype(str).str.strip().str.upper() == 'SIM')
    & (f4['tem_dano_atlas'] == True)
].copy()

print('\n=== MÉTRICA DO FUNIL ATUALIZADA ===')
print(
    f'Total Inicial: {len(f0)} | SINISA: {len(f1)} | Com Risco: {len(f2)} |'
    f' CAPAG C/D/Sem Nota: {len(f3)} | Mapeados: {len(f4)} |'
    f' Elegíveis Finais (Crítico + dano no Atlas): {len(f5)}'
)


=== MÉTRICA DO FUNIL ATUALIZADA ===
Total Inicial: 5570 | SINISA: 4845 | Com Risco: 2238 | CAPAG C/D/Sem Nota: 1032 | Mapeados: 622 | Elegíveis Finais (Crítico + dano no Atlas): 318


**Ordenação e Construção do Raking - TOP 20** -
(OPÇÃO 70% QTD / 30% PCT)

In [3]:
f5['rank_qtd_pctile'] = f5['qtd_domicilios_risco'].rank(pct=True)
f5['rank_pct_pctile'] = f5['pct_domicilios_risco'].rank(pct=True)
f5['nota_70_30'] = (0.7 * f5['rank_qtd_pctile']) + (0.3 * f5['rank_pct_pctile'])

top_20 = f5.nlargest(20, 'nota_70_30').copy().reset_index(drop=True)
top_20['posicao'] = top_20.index + 1

In [4]:
# Adicionando uma coluna de posição utilizando a variável correta (top_20).
top_20['posicao'] = top_20.index + 1

top_20.head(20)

,Cod_IBGE_7,Nome,UF,mapeamento_area_risco,qtd_domicilios_risco,municipio_critico,pct_vias_rede_drenagem,investimento_percapita_drenagem,pct_domicilios_risco,participou_sinisa,...,soma_danos_materiais,soma_prejuizo_publico,danos_humanos,tem_registro_atlas,tem_dano_atlas,capag_limpa,rank_qtd_pctile,rank_pct_pctile,nota_70_30,posicao
0,4307708,Esteio,RS,Sim,30000.0,Sim,98.63,0.00,88.18,True,...,20477184.43,2908522.97,20814.0,True,True,C,0.984277,0.996855,0.988050,1
1,4304606,Canoas,RS,Sim,70000.0,Sim,96.81,166.99,43.53,True,...,6910085.69,7025897.78,184680.0,True,True,C,0.993711,0.968553,0.986164,2
2,3304904,São Gonçalo,RJ,Sim,175838.0,Sim,5.60,39.73,40.00,True,...,27405589.64,3075978.70,624.0,True,True,C,1.000000,0.943396,0.983019,3
3,5002704,Campo Grande,MS,Sim,125000.0,Sim,40.84,7.70,30.43,True,...,112000.00,150000.00,0.0,True,True,C,0.996855,0.905660,0.969497,4
4,3301009,Campos dos Goytacazes,RJ,Sim,63000.0,Sim,9.37,32.59,27.21,True,...,26239841.06,0.00,1142.0,True,True,C,0.990566,0.880503,0.957547,5
5,3303708,Paraíba do Sul,RJ,Sim,8561.0,Sim,45.45,0.00,46.47,True,...,3706249.92,413489.66,1016.0,True,True,C,0.943396,0.981132,0.954717,6
6,2601409,Barreiros,PE,Sim,7938.0,Sim,6.78,0.00,50.00,True,...,667387.98,67359.25,7193.0,True,True,C,0.933962,0.990566,0.950943,7
7,3156700,Sabará,MG,Sim,16886.0,Sim,40.00,45.41,29.70,True,...,13145753.76,1307908.03,2639.0,True,True,C,0.965409,0.893082,0.943711,8
8,2610004,Palmares,PE,Sim,8000.0,Sim,NaN,NaN,40.49,True,...,15709645.23,5368592.42,963.0,True,True,C,0.938679,0.949686,0.941981,9
9,1600279,Laranjal do Jari,AP,Sim,6000.0,Sim,18.90,0.00,45.01,True,...,81766.53,134894.41,388.0,True,True,N.E.,0.921384,0.977987,0.938365,10


DISTRIBUIÇÃO DA VERBA - OPÇÃO B (PROPORCIONAL TAMANHO DO MUNICÍPIO)

In [5]:
VERBA_TOTAL = 150_000_000  # R$ 150 Milhões
PISO_MINIMO = 3_000_000  # R$ 3 Milhões (Piso)
TETO_MAXIMO = 15_000_000  # R$ 15 Milhões (Teto)

# Divisão proporcional às moradias com piso e teto (Decisão 4B):
# procura o valor por moradia (r) tal que a soma de min(max(r × moradias, piso), teto) dê exatamente R$ 150 mi.
# Assim todo município recebe o mesmo valor por moradia, exceto os que batem no piso ou no teto
# (e ninguém passa do teto nem fica abaixo do piso).
moradias = top_20['qtd_domicilios_risco']
lo, hi = 0.0, TETO_MAXIMO / moradias.min()
for _ in range(100):
  r = (lo + hi) / 2
  if (r * moradias).clip(PISO_MINIMO, TETO_MAXIMO).sum() < VERBA_TOTAL:
    lo = r
  else:
    hi = r
top_20['verba_ajustada'] = (r * moradias).clip(lower=PISO_MINIMO, upper=TETO_MAXIMO)

top_20['verba_alocada_m'] = (top_20['verba_ajustada'] / 1_000_000).round(2)

# Mapeamento do perfil fiscal
top_20['situacao_fiscal'] = (
    top_20['capag_limpa']
    .map({
        'C': 'Fragilidade Fiscal',
        'D': 'Alta Fragilidade Fiscal',
        'N.D.': 'Pendência Contábil (Sem Nota)',
        'N.E.': 'Não Elegível (Sem Nota)',
    })
    .fillna('Sem Nota/Pendente')
)

**Corte e cálculo "custo de não agir"**

In [6]:
# Métrica de cobertura do orçamento
demais_elegiveis = f5[~f5['Cod_IBGE_7'].isin(top_20['Cod_IBGE_7'])].copy()

# Garantir conversão numérica das colunas do Atlas de Desastres
cols_atlas = ['total_desastres', 'soma_danos_materiais', 'danos_humanos']
for col in cols_atlas:
  top_20[col] = pd.to_numeric(top_20[col], errors='coerce').fillna(0)
  demais_elegiveis[col] = (
      pd.to_numeric(demais_elegiveis[col], errors='coerce').fillna(0)
  )

tabela_custo_nao_agir = pd.DataFrame([
    {
        'Grupo': '20 Selecionados (Top 20)',
        'Qtd Cidades': len(top_20),
        'Moradias em Risco (SINISA)': (
            f"{top_20['qtd_domicilios_risco'].sum():,.0f}".replace(',', '.')
        ),
        'Total de Desastres (2021-2025)': (
            f"{top_20['total_desastres'].sum():,.0f}".replace(',', '.')
        ),
        'Danos Materiais Totais (R$)': (
            f"R$ {top_20['soma_danos_materiais'].sum():,.2f}".replace(
                ',', 'X'
            )
            .replace('.', ',')
            .replace('X', '.')
        ),
        'Pessoas Afetadas (Danos Humanos)': (
            f"{top_20['danos_humanos'].sum():,.0f}".replace(',', '.')
        ),
    },
    {
        'Grupo': 'Demais Elegíveis (Fora do Top 20)',
        'Qtd Cidades': len(demais_elegiveis),
        'Moradias em Risco (SINISA)': (
            f"{demais_elegiveis['qtd_domicilios_risco'].sum():,.0f}".replace(
                ',', '.'
            )
        ),
        'Total de Desastres (2021-2025)': (
            f"{demais_elegiveis['total_desastres'].sum():,.0f}".replace(
                ',', '.'
            )
        ),
        'Danos Materiais Totais (R$)': (
            f"R$ {demais_elegiveis['soma_danos_materiais'].sum():,.2f}".replace(
                ',', 'X'
            )
            .replace('.', ',')
            .replace('X', '.')
        ),
        'Pessoas Afetadas (Danos Humanos)': (
            f"{demais_elegiveis['danos_humanos'].sum():,.0f}".replace(',', '.')
        ),
    },
])

print('=== TABELA: CUSTO DE NÃO AGIR ===')
display(tabela_custo_nao_agir)

=== TABELA: CUSTO DE NÃO AGIR ===


,Grupo,Qtd Cidades,Moradias em Risco (SINISA),Total de Desastres (2021-2025),Danos Materiais Totais (R$),Pessoas Afetadas (Danos Humanos)
0,20 Selecionados (Top 20),20,635.147,153,"R$ 571.455.397,00",283.377
1,Demais Elegíveis (Fora do Top 20),298,381.366,1.280,"R$ 6.691.808.868,50",680.773


Texte de Consistências

In [7]:
moradias_atendidas = top_20['qtd_domicilios_risco'].sum()
moradias_totais_elegiveis = f5['qtd_domicilios_risco'].sum()
pct_cobertura = (moradias_atendidas / moradias_totais_elegiveis) * 100

print('\n=== TESTES DE CONSISTÊNCIA E VALIDAÇÃO DE REGRAS ===')

# Teste 1: Validação do Funil e Mapeamento
mapeamento_ok = (top_20['mapeamento_area_risco'] == 'Sim').sum()
print(
    f'✅ Teste 1 (Mapeamento de Risco): {mapeamento_ok} de 20 municípios no'
    ' Top 20 possuem Mapeamento de Risco confirmado.'
)

# Teste 2: Validação da Regra CAPAG (Exclusão de A, A+, B, B+)
capag_invalidas = top_20['capag_limpa'].isin(['A', 'A+', 'B', 'B+']).sum()
if capag_invalidas == 0:
  print(
      '✅ Teste 2 (Filtro Fiscal): 100% dos municípios no Top 20 possuem'
      ' restrição de acesso a crédito (CAPAG C, D ou Sem Nota).'
  )

# Teste 3: Cobertura de Moradias no Orçamento
print(
    f'✅ Teste 3 (Cobertura de Risco): O Top 20 protege'
    f' {moradias_atendidas:,.0f} moradias de um total de'
    f' {moradias_totais_elegiveis:,.0f} nos elegíveis ({pct_cobertura:.1f}% de'
    ' cobertura).'
)

# Teste 4: Fechamento Orçamentário (Verba = R$ 150 Mi)
verba_fechamento = top_20['verba_ajustada'].sum()
print(
    f'✅ Teste 4 (Fechamento Orçamentário): Soma total alocada no Top 20 = R$'
    f' {verba_fechamento/1e6:.2f} Mi (Meta: R$ 150,00 Mi).'
)

# Teste 5: Piso e teto respeitados (R$ 3 mi a R$ 15 mi por município)
fora_limite = (~top_20['verba_ajustada'].between(PISO_MINIMO - 1, TETO_MAXIMO + 1)).sum()
assert fora_limite == 0, f'{fora_limite} município(s) fora do piso/teto'
print(
    f'✅ Teste 5 (Piso e Teto): todos entre R$ 3 mi e R$ 15 mi; de R$'
    f" {(top_20['verba_ajustada'] / top_20['qtd_domicilios_risco']).min():.0f} a R$"
    f" {(top_20['verba_ajustada'] / top_20['qtd_domicilios_risco']).max():.0f} por moradia."
)


=== TESTES DE CONSISTÊNCIA E VALIDAÇÃO DE REGRAS ===
✅ Teste 1 (Mapeamento de Risco): 20 de 20 municípios no Top 20 possuem Mapeamento de Risco confirmado.
✅ Teste 2 (Filtro Fiscal): 100% dos municípios no Top 20 possuem restrição de acesso a crédito (CAPAG C, D ou Sem Nota).
✅ Teste 3 (Cobertura de Risco): O Top 20 protege 635,147 moradias de um total de 1,016,513 nos elegíveis (62.5% de cobertura).
✅ Teste 4 (Fechamento Orçamentário): Soma total alocada no Top 20 = R$ 150.00 Mi (Meta: R$ 150,00 Mi).
✅ Teste 5 (Piso e Teto): todos entre R$ 3 mi e R$ 15 mi; de R$ 85 a R$ 690 por moradia.


EXPORTAÇÃO DOS ARQUIVOS PARA O LOOKER STUDIO

In [8]:
colunas_export = [
    'posicao',
    'Cod_IBGE_7',
    'Nome',
    'UF',
    'capag',
    'situacao_fiscal',
    'qtd_domicilios_risco',
    'verba_alocada_m',
    'total_desastres',
    'danos_humanos',
    'soma_danos_materiais',
]

tabela_export = top_20[colunas_export].copy()

# Exportação para CSV e Excel
csv_nome = 'Painel_v1_Top20_OpcaoB_ComSemNota.csv'
xlsx_nome = 'Painel_v1_Top20_OpcaoB_ComSemNota.xlsx'

tabela_export.to_csv(csv_nome, index=False, encoding='utf-8-sig')
tabela_export.to_excel(xlsx_nome, index=False)

print(
    f"\n🚀 Sucesso! Arquivos '{csv_nome}' e '{xlsx_nome}' prontos para o Looker"
    ' Studio.'
)

# Baixa os arquivos quando roda no Colab (fora dele, ficam salvos na pasta do notebook)
try:
  from google.colab import files
  files.download(csv_nome)
  files.download(xlsx_nome)
except ImportError:
  pass


🚀 Sucesso! Arquivos 'Painel_v1_Top20_OpcaoB_ComSemNota.csv' e 'Painel_v1_Top20_OpcaoB_ComSemNota.xlsx' prontos para o Looker Studio.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>